# II. Handling Chemistry Data

Pre-modelling data interrogation for reaction datasets.
This notebook focuses on chemical sanity checks, distributions, and simple trend analysis before any modelling.


## 0. Learning objectives

By the end of this notebook you will be able to:

- Load real reaction datasets
- Identify target, categorical, and continuous variables
- Detect missing, corrupted, or non-physical values
- Explore yield distributions and chemical trends
- Perform simple regression and correlation checks
- Produce a modelling-ready table (datasets here are already clean enough for modelling)


## 1. Dataset selection

Choose a reaction dataset to explore individually or as a group. The worked examples use the supplied datasets; adapting another dataset may require changes to its features and oracle. Large categorical spaces can make optimization more difficult. See [data/README.md](data/README.md) for source citations, preprocessing, and reuse information.

- `Dataset_A_Aspuru-Guzik_2021`: [paper_A](https://iopscience.iop.org/article/10.1088/2632-2153/abedc8) | [data_A](https://github.com/the-matter-lab/olympus/tree/main/src/olympus/datasets/dataset_suzuki) | [docs_A](https://aspuru-guzik-group.github.io/olympus/classes/datasets/index.html) | local `Dataset_A1`
- `Dataset_B_Jensen_2016`: [paper_B](https://doi.org/10.1039/C6RE00153J) | [data_B](https://www.rsc.org/suppdata/c6/re/c6re00153j/c6re00153j1.pdf) | local `Dataset_B1` / `Dataset_B2` / `Dataset_B3` / `Dataset_B4`
- `Dataset_C_Newman_2021`: [paper_C](https://www.nature.com/articles/s41596-020-00452-7) | [data_C](https://static-content.springer.com/esm/art%3A10.1038%2Fs41596-020-00452-7/MediaObjects/41596_2020_452_MOESM1_ESM.pdf) | local `Dataset_C1`
- `Dataset_D_Doyle_2021`: [paper_D](https://www.nature.com/articles/s41586-021-03213-y) | [data_D](https://github.com/b-shields/edbo/tree/master/experiments/data) | local `Dataset_D1`
- `Dataset_E_Bourne_2023`: [paper_E](https://www.sciencedirect.com/science/article/pii/S1385894722039250#s0060) | local `Dataset_E1` / `Dataset_E2`
- Frequently, the works of the following groups contain similar data: Abigail Doyle, Richard Bourne, Alan Aspuru-Guzik, Klavs Jensen, Martin Burke and Scott Denmark

Current local files in `data/`:

- `Dataset_A1_Aspuru-Guzik_Suzuki_Numeric` (numeric-only; [csv_A1](data/dataset_a1_aspuru_guzik_suzuki_numeric.csv))
- `Dataset_B1_Jensen_Case_I` (Table S.1 / Reaction of 1 and 2; [csv_B1](data/dataset_b1_jensen_case_i.csv))
- `Dataset_B2_Jensen_Case_II` (Table S.3 / Reaction of 4 and 2; [csv_B2](data/dataset_b2_jensen_case_ii.csv))
- `Dataset_B3_Jensen_Case_III` (Table S.5 / Reaction of 4 and 6; [csv_B3](data/dataset_b3_jensen_case_iii.csv))
- `Dataset_B4_Jensen_Case_IV` (Paper B case IV subset; [csv_B4](data/dataset_b4_jensen_case_iv.csv))
- `Dataset_C1_Newman_BH_Plate_Replicates` (Supporting Table 1 / Plates A-C; [csv_C1](data/dataset_c1_newman_bh_plate_replicates.csv))
- `Dataset_D1_Doyle_Direct_Arylation` (reaction 3 main set; [csv_D1](data/dataset_d1_doyle_direct_arylation.csv))
- `Dataset_E1_Bourne_SNAr` (Table S5 with Pareto flag from Table S6; [csv_E1](data/dataset_e1_bourne_snar.csv))
- `Dataset_E2_Bourne_Sonogashira` (Table S7 with Pareto flag from Table S8; [csv_E2](data/dataset_e2_bourne_sonogashira.csv))
- `Dataset_Legend` (label map; [csv_legend](data/dataset_legend.csv))
- `Dataset_Summary` (inputs/outputs map; [csv_summary](data/dataset_summary.csv))

`DATASET_KEY` below currently switches between `Dataset_B4` and `Dataset_A1`. Default: `Dataset_B4_Jensen_Case_IV`.


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LinearRegression

DATA_DIR = Path("data")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)


In [ ]:
# Choose dataset
DATASET_KEY = "dataset_b4"  # Options: 'dataset_b4', 'dataset_a1'

if DATASET_KEY == "dataset_b4":
    # Categorical + multi-target example
    dataset_name = "Dataset_B4_Jensen_Case_IV (categorical + multi-target)"
    source = DATA_DIR / "dataset_b4_jensen_case_iv.csv"
    df = pd.read_csv(source)
    # Assign meaningful column names for readability
    df.columns = [
        "ligand",
        "res_time",
        "temperature",
        "catalyst_loading",
        "yield",
        "turnover",
    ]

    target_cols = ["yield", "turnover"]
    # Discrete categories (ligand, solvent, base, etc.)
    categorical_cols = ["ligand"]
    # Continuous, measurable reaction conditions (temperature, time, loading, etc.)
    continuous_cols = ["res_time", "temperature", "catalyst_loading"]
    # Experimental bounds used to flag non-physical values
    experimental_bounds = {
        "yield": (0, 100),
        "turnover": (0, 1000),
        "temperature": (0, 200),
        "res_time": (0, 2000),
        "catalyst_loading": (0, 10),
    }
elif DATASET_KEY == "dataset_a1":
    # Numeric-only example
    dataset_name = "Dataset_A1_Aspuru-Guzik_Suzuki_Numeric (numeric-only)"
    source = DATA_DIR / "dataset_a1_aspuru_guzik_suzuki_numeric.csv"
    df = pd.read_csv(source)
    # Assign meaningful column names for readability
    df.columns = [
        "temperature_C",
        "Pd_mol_percent",
        "ArBpin_equiv",
        "K3PO4_equiv",
        "yield_percent",
    ]

    target_cols = ["yield_percent"]
    categorical_cols = []
    # Continuous, measurable reaction conditions (temperature, time, loading, etc.)
    continuous_cols = ["temperature_C", "Pd_mol_percent", "ArBpin_equiv", "K3PO4_equiv"]
    # Experimental bounds used to flag non-physical values
    experimental_bounds = {
        "yield_percent": (0, 100),
        "temperature_C": (50, 120),
        "Pd_mol_percent": (0, 10),
        "ArBpin_equiv": (0, 3),
        "K3PO4_equiv": (0, 5),
    }
else:
    raise ValueError("Unknown DATASET_KEY; use 'dataset_b4' or 'dataset_a1'")

print(f"Using dataset: {dataset_name}")
df.head()


## 2. Raw data ingestion and first look

Inspect shape, column names, and datatypes before any analysis.


In [ ]:
df.shape


In [ ]:
df.columns


In [ ]:
df.dtypes


### Optional: browse in Excel

It can be a good idea to visually scan raw data in a spreadsheet before analysis.
Export a CSV to open in Excel or another spreadsheet application.


In [ ]:
# df.head(200).to_csv(f"preview_{DATASET_KEY}.csv", index=False)


## 3. Structural understanding of the dataset

Identify which columns are categorical, continuous, and target variables. This is to ensure that the data is encoded properly, as by themselves, these datasets do not idenitfy which columns are which.


In [ ]:
target_col = target_cols[0]
feature_cols = categorical_cols + continuous_cols

print("Targets:", target_cols)
print("Categorical features:", categorical_cols)
print("Numeric features:", continuous_cols)


In [ ]:
# Quick summary stats
df[continuous_cols + target_cols].describe().T


In [ ]:
# Cardinality of categorical variables
if categorical_cols:
    df[categorical_cols].nunique().sort_values()
else:
    print("No categorical columns in this dataset.")


## 4. Chemical sanity checks

Check for missing values, impossible values, and duplicated experiments.


In [ ]:
# Missing values per column
df.isna().sum().sort_values(ascending=False).head(20)


In [ ]:
# Duplicate rows
df.duplicated().sum()


In [ ]:
# Out-of-range / non-physical checks
def out_of_range_rows(dataframe, bounds):
    mask = pd.Series(False, index=dataframe.index)
    for col, (lo, hi) in bounds.items():
        if col in dataframe.columns:
            mask |= dataframe[col].lt(lo) | dataframe[col].gt(hi)
    return dataframe[mask]


invalid_rows = out_of_range_rows(df, experimental_bounds)
invalid_rows.head()


## 5. Distributional chemistry

Look at yield distributions and how they vary across categorical variables. Ensuring balanced coverage across categorical variables reduces sampling bias and improves the uniformity of model support across the search space.

In [ ]:
plt.figure(figsize=(6, 4))
df[target_col].hist(bins=30)
plt.xlabel(target_col)
plt.ylabel("Count")
plt.title("Yield distribution")
plt.show()


In [ ]:
# Quantiles for outlier detection
df[target_col].quantile([0.01, 0.05, 0.95, 0.99])


In [ ]:
# Boxplot by top categories (if available)
if categorical_cols:
    cat = categorical_cols[0]
    top = df[cat].value_counts().head(8).index
    subset = df[df[cat].isin(top)]
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=subset, x=cat, y=target_col)
    plt.xticks(rotation=45, ha="right")
    plt.title(f"{target_col} by {cat} (top categories)")
    plt.show()
else:
    print("No categorical columns to plot.")


## 6. Pairwise chemical trends

Scatter plots provide a fast way to identify relationships between variables, as well as anomalies before applying more complex modelling.

In [ ]:
# Trend plots for continuous features
# PR: https://seaborn.pydata.org/generated/seaborn.pairplot.html
for col in continuous_cols:
    if col in df.columns:
        plt.figure(figsize=(5, 4))
        plt.scatter(df[col], df[target_col], alpha=0.5)
        plt.xlabel(col)
        plt.ylabel(target_col)
        plt.title(f"{target_col} vs {col}")
        plt.show()


## 7. Simple Linear Regression

### Overview
If simple trends are observable, then this can be easily explained by individual 1-D linear regressions.  

Note that **linear regression refers to the form of the function (specifically the coefficients)** rather than the number of variables. For example, 2-D linear regression would be the same as fitting a 2-D plane in 3-D space.

---

### Types of Linear Regression Considered

#### 1. 1-D Linear Regression
- Applied to individual variables  
- Captures simple, direct trends between a single input and the output  

#### 2. Global (Multiple) Linear Regression
- Considers all inputs simultaneously  
- Gives the simplest form of equation relating inputs to outputs  

---

### Interpretation and Use

- Both serve as a **(pretty terrible) baseline** for comparing models, such as GPs  
- Can be considered a **surrogate model**  
- Conceptually acts as a **stepping stone towards DoE**

---

### Connection to DoE

- Global multiple linear regression is equivalent to a **first-order DoE model**
- Where:
  - Exclusively **main effects** are used
  - **No interaction or higher-order terms**
  -  No terms are added or removed  


In [ ]:
def one_dimensional_linear_regressions(df, x_cols, y_col):
    """
    Run individual 1-D linear regressions:
    yield ≈ a·x + b for each x in x_cols

    Returns dictionary of fitted models and R² diagnostics.
    """

    results = {}

    for x in x_cols:
        sub = df[[x, y_col]].dropna()
        if len(sub) < 5:
            print(f"Skipping {x} (not enough data)")
            continue

        X = sub[[x]]
        y = sub[y_col]

        model = LinearRegression().fit(X, y)
        r2 = model.score(X, y)

        print(f"\nYield ≈ a·{x} + b")
        print(f"a (Δ yield/Δ {x}): {model.coef_[0]:.4f}")
        print(f"b (Intercept): {model.intercept_:.2f}")
        print(f"R²({x}): {r2:.3f}")

        plt.figure()
        plt.scatter(X, y)
        plt.plot(X, model.predict(X))
        plt.xlabel(x)
        plt.ylabel(y_col)
        plt.title(f"{y_col} vs {x}")
        plt.show()

        results[x] = {
            "model": model,
            "coef": model.coef_[0],
            "intercept": model.intercept_,
            "r2": r2,
        }

    return results


def global_linear_regression(df, x_cols, y_col):
    """
    Fits a full multivariate linear model:

        y ≈ β0 + Σ βi·xi

    Returns coefficient table, R², and residual diagnostics.
    """

    sub = df[x_cols + [y_col]].dropna()

    X = sub[x_cols]
    y = sub[y_col]

    model = LinearRegression().fit(X, y)
    r2 = model.score(X, y)

    print("\nGLOBAL LINEAR MODEL")
    print("y ≈ β0 + Σ βi·xi\n")

    for col, coef in zip(x_cols, model.coef_):
        print(f"β_{col} (Δy/Δ{col}): {coef:.4f}")

    print(f"\nβ0 (Intercept): {model.intercept_:.2f}")
    print(f"GLOBAL R²: {r2:.3f}")

    # Residual diagnostics
    residuals = y - model.predict(X)
    rmse = np.sqrt(np.mean(residuals**2))

    print(f"RMSE: {rmse:.2f}")

    return {
        "model": model,
        "target_col": y_col,
        "x_cols": x_cols,
        "r2": r2,
        "rmse": rmse,
        "coefficients": dict(zip(x_cols, model.coef_)),
        "intercept": model.intercept_,
        "residuals": residuals,
    }


one_dimensional_regression_results = one_dimensional_linear_regressions(
    df, continuous_cols, target_col
)


In [ ]:
def predict_target_from_global_linear_regression(
    conditions: dict, global_model: dict, clip_to_bounds=True
):
    """
    Deterministically compute predicted target (e.g. yield)
    from raw reaction conditions using a fitted global linear surrogate.
    """

    y_hat = global_model["intercept"]

    # Linear sum
    for col, coef in global_model["coefficients"].items():
        y_hat += coef * float(conditions.get(col, 0))

    # Optional physical clipping (self-contained)
    target = global_model["target_col"]
    if clip_to_bounds and target in experimental_bounds:
        lo, hi = experimental_bounds[target]
        y_hat = float(np.clip(y_hat, lo, hi))

    return y_hat


global_yield_model = global_linear_regression(df, continuous_cols, "yield")

conditions = {"res_time": 120, "temperature": 100, "catalyst_loading": 2.5}

print("Predicting yield for: " + ", ".join(f"{k}={v}" for k, v in conditions.items()))
print(
    f"Yield = {predict_target_from_global_linear_regression(conditions, global_yield_model)}"
)


## 8. Correlation heatmap

Check which continuous variables correlate with yield.


In [ ]:
corr_cols = [c for c in continuous_cols + [target_col] if c in df.columns]
corr = df[corr_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation heatmap")
plt.show()


## 9. Categorical performance ranking

Rank ligands by mean yield.


In [ ]:
# Rank a categorical variable by average target performance
if categorical_cols:
    rank_col = categorical_cols[0]
    ranking = (
        df.groupby(rank_col)[target_col]
        .agg(["mean", "std", "count"])
        .sort_values("mean", ascending=False)
    )
    ranking.head(10)
else:
    print("No categorical columns available for ranking.")


## 10. Chemical sub-space exploration

Zoom in on a single ligand and re-plot trends locally.


In [ ]:
if categorical_cols:
    rank_col = categorical_cols[0]
    focus_value = df[rank_col].value_counts().index[0]
    subset = df[df[rank_col] == focus_value]
    print(f"Focus: {rank_col} = {focus_value} (n={len(subset)})")

    for col in continuous_cols:
        if col in subset.columns:
            plt.figure(figsize=(5, 4))
            plt.scatter(subset[col], subset[target_col], alpha=0.6)
            plt.xlabel(col)
            plt.ylabel(target_col)
            plt.title(f"{target_col} vs {col} at {rank_col}={focus_value}")
            plt.show()
else:
    print("No categorical columns available for sub-space exploration.")


## 11. Modelling readiness note

The datasets used here are already curated and clean enough for modelling.
In real projects you should still document any cleaning steps, but we skip them here to keep the focus on chemical interrogation.


## 12. Reflection questions

In judging the quality of data, you can use these questions to reflect on its suitability for modelling.

- Are trends apparent already or not?
- Which variable appears most important chemically?
- Which variables are misleading or noisy?
- Are there conflicting trends across different ligands?
- Where is the dataset chemically weak or under-sampled?

In uncurated data, it's important that the data is thoroughly checked and analysed to prevent anomalies and inconsistencies from affecting modeling, or preventing it altogether.
